In [11]:
import numpy as np
import pandas as pd
import yfinance as yf
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime, timedelta



In [12]:

# Download data
stock = "GOOG"
start = "2012-01-01"
end = (datetime.today() - timedelta(days=1)).strftime('%Y-%m-%d')

data = yf.download(stock, start, end)


[*********************100%***********************]  1 of 1 completed


In [13]:
data = data[['Close']]
split_index = int(len(data) * 0.8)

data_train = data[:split_index]
data_test = data[split_index:]


In [14]:

scaler = MinMaxScaler(feature_range=(0,1))
data_train_scaled = scaler.fit_transform(data_train.values)


In [15]:

x = []
y = []

for i in range(100, data_train_scaled.shape[0]):
    x.append(data_train_scaled[i-100:i])
    y.append(data_train_scaled[i,0])

x = np.array(x)
y = np.array(y)

x = torch.tensor(x, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).view(-1,1)


In [16]:

# LSTM Model
class StockLSTM(nn.Module):

    def __init__(self):
        super().__init__()

        self.lstm1 = nn.LSTM(1,50,batch_first=True)
        self.dropout1 = nn.Dropout(0.2)

        self.lstm2 = nn.LSTM(50,60,batch_first=True)
        self.dropout2 = nn.Dropout(0.3)

        self.lstm3 = nn.LSTM(60,80,batch_first=True)
        self.dropout3 = nn.Dropout(0.4)

        self.lstm4 = nn.LSTM(80,120,batch_first=True)
        self.dropout4 = nn.Dropout(0.5)

        self.fc = nn.Linear(120,1)

    def forward(self,x):

        x,_ = self.lstm1(x)
        x = self.dropout1(x)

        x,_ = self.lstm2(x)
        x = self.dropout2(x)

        x,_ = self.lstm3(x)
        x = self.dropout3(x)

        x,_ = self.lstm4(x)
        x = self.dropout4(x)

        x = x[:,-1,:]
        x = self.fc(x)

        return x




In [17]:
model = StockLSTM()

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

epochs = 50
batch_size = 32

best_loss = float('inf')

for epoch in range(epochs):
    # shuffle each epoch
    perm = torch.randperm(len(x))
    x_shuffled = x[perm]
    y_shuffled = y[perm]

    for i in range(0, len(x_shuffled), batch_size):
        batch_x = x_shuffled[i:i+batch_size]
        batch_y = y_shuffled[i:i+batch_size]

        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

    if loss.item() < best_loss:
        best_loss = loss.item()
        torch.save(model.state_dict(), "stock_model.pth")
        print(f"Epoch {epoch+1}/{epochs} Loss: {loss.item():.6f} ✅ saved")
    else:
        print(f"Epoch {epoch+1}/{epochs} Loss: {loss.item():.6f}")

print(f"Training done! Best loss: {best_loss:.6f}")

torch.save(model.state_dict(), "stock_model.pth")
print("Model trained and saved as stock_model.pth")

Epoch 1/50 Loss: 0.079441 ✅ saved
Epoch 2/50 Loss: 0.021332 ✅ saved
Epoch 3/50 Loss: 0.005547 ✅ saved
Epoch 4/50 Loss: 0.002955 ✅ saved
Epoch 5/50 Loss: 0.003245
Epoch 6/50 Loss: 0.002943 ✅ saved
Epoch 7/50 Loss: 0.002888 ✅ saved
Epoch 8/50 Loss: 0.007838
Epoch 9/50 Loss: 0.003983
Epoch 10/50 Loss: 0.003230
Epoch 11/50 Loss: 0.002357 ✅ saved
Epoch 12/50 Loss: 0.002798
Epoch 13/50 Loss: 0.001213 ✅ saved
Epoch 14/50 Loss: 0.001869
Epoch 15/50 Loss: 0.004413
Epoch 16/50 Loss: 0.003360
Epoch 17/50 Loss: 0.000953 ✅ saved
Epoch 18/50 Loss: 0.001334
Epoch 19/50 Loss: 0.003134
Epoch 20/50 Loss: 0.001884
Epoch 21/50 Loss: 0.000991
Epoch 22/50 Loss: 0.002417
Epoch 23/50 Loss: 0.002664
Epoch 24/50 Loss: 0.003848
Epoch 25/50 Loss: 0.002471
Epoch 26/50 Loss: 0.001602
Epoch 27/50 Loss: 0.003945
Epoch 28/50 Loss: 0.001629
Epoch 29/50 Loss: 0.002257
Epoch 30/50 Loss: 0.001238
Epoch 31/50 Loss: 0.002031
Epoch 32/50 Loss: 0.001700
Epoch 33/50 Loss: 0.001969
Epoch 34/50 Loss: 0.001373
Epoch 35/50 Loss: 0